# Question: Writing Viterbi Algorithm for the Primer
# Author: G Mahendra Reddy
# Enrollment Number : 23114030

In [5]:
# Define the HMM parameters
states = ['E', '5', 'I']
hmm_transitions = {
    'Start': {'E': 1.0},
    'E': {'E': 0.9, '5': 0.1},
    '5': {'I': 1.0},
    'I': {'I': 0.9, 'End': 0.1}
}

hmm_emissions = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
    '5': {'A': 0.05, 'C': 0.0,  'G': 0.95, 'T': 0.0},
    'I': {'A': 0.4,  'C': 0.1,  'G': 0.1,  'T': 0.4}
}

In [6]:
import math

def safe_log(value):
    """Compute the natural log of a value, returning -inf if the value is zero."""
    return -math.inf if value == 0 else math.log(value)

def compute_log_probability(state_path, observed_sequence):
    """
    Calculate the log-probability of an observed sequence following a given state path
    using the defined HMM transition and emission probabilities.
    """
    if len(state_path) != len(observed_sequence):
        raise ValueError("State path and observed sequence must be of the same length.")

    total_log_prob = 0.0
    previous_state = 'Start'

    for i in range(len(observed_sequence)):
        current_state = state_path[i]
        observed_symbol = observed_sequence[i]

        transition_probability = hmm_transitions[previous_state][current_state]
        emission_probability = hmm_emissions[current_state][observed_symbol]

        total_log_prob += safe_log(transition_probability) + safe_log(emission_probability)
        previous_state = current_state

    # Add the transition from the last state to 'End' if necessary
    if previous_state == 'I':
        total_log_prob += safe_log(hmm_transitions[previous_state]['End'])

    return round(total_log_prob, 2)



# Test on example input
hidden_path = "EEEEEEEEEEEEEEEEEE5IIIIIII"
observed_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"

# Compute and display the log-probability
log_probability = compute_log_probability(hidden_path, observed_sequence)
print(f"Log-probability of the given path: {log_probability}")


Log-probability of the given path: -41.22


In [7]:
import math

def safe_log(x):
    # Returns log(x), or -inf if x is 0 to avoid math domain errors.
    return -math.inf if x == 0 else math.log(x)

def run_viterbi(observed_seq):
    """
    Executes the Viterbi algorithm to determine the most probable sequence of hidden states
    for a given DNA sequence based on defined HMM transition and emission probabilities.
    """
    sequence_length = len(observed_seq)
    viterbi_table = [{}]  # Stores log probabilities at each time step
    state_tracker = {}    # Tracks the optimal path for each state

    # Initialization step: start from state 'E'
    for state in ['E']:
        trans_prob = hmm_transitions['Start'][state]
        emit_prob = hmm_emissions[state][observed_seq[0]]
        viterbi_table[0][state] = safe_log(trans_prob) + safe_log(emit_prob)
        state_tracker[state] = [state]

    # Fill in the Viterbi table
    for t in range(1, sequence_length):
        viterbi_table.append({})
        new_tracker = {}

        for curr_state in hmm_states:
            max_score = -math.inf
            best_prev_state = None

            for prev_state in viterbi_table[t - 1]:
                if curr_state in hmm_transitions.get(prev_state, {}):
                    trans_log = safe_log(hmm_transitions[prev_state][curr_state])
                    emit_log = safe_log(hmm_emissions[curr_state][observed_seq[t]])
                    score = viterbi_table[t - 1][prev_state] + trans_log + emit_log

                    if score > max_score:
                        max_score = score
                        best_prev_state = prev_state

            if best_prev_state:
                viterbi_table[t][curr_state] = max_score
                new_tracker[curr_state] = state_tracker[best_prev_state] + [curr_state]

        state_tracker = new_tracker

    # Termination: select the best final state
    final_state = max(viterbi_table[-1], key=viterbi_table[-1].get)
    final_score = viterbi_table[-1][final_state]
    best_path = ''.join(state_tracker[final_state])

    return best_path, round(final_score, 2)

# Define the HMM components
hmm_states = ['E', '5', 'I']
hmm_transitions = {
    'Start': {'E': 1.0},
    'E': {'E': 0.9, '5': 0.1},
    '5': {'I': 1.0},
    'I': {'I': 0.9, 'End': 0.1}
}
hmm_emissions = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
    '5': {'A': 0.05, 'C': 0.00, 'G': 0.95, 'T': 0.00},
    'I': {'A': 0.40, 'C': 0.10, 'G': 0.10, 'T': 0.40}
}

# Input sequence
dna_input = "CTTCATGTGAAAGCAGACGTAAGTCA"

# Run the Viterbi algorithm
optimal_path, log_probability = run_viterbi(dna_input)

print("Most likely path for the given sequence:", optimal_path)
print("Log probability of the best path:", log_probability)


Most likely path for the given sequence: EEEEEEEEEEEEEEEEEEEEEEEEEE
Log probability of the best path: -38.68
